In [1]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pickle
from pathlib import Path

import matplotlib.pyplot as plt

from models_transformer import SingleOutTransformerNet
from attention_rollout_helpers import compute_rollout_over_dataloader, get_batch_rollout
from attention_rollout_helpers import get_rollout_importance
from attention_rollout_helpers import extract_attention, process_attention

### Load data

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_data = np.load("data_processed/test_data_scaled.npz")
X_test = test_data["x"]    
y_test = test_data["age"]    
test_loader = DataLoader(X_test, batch_size=32, shuffle=False)

In [3]:
SEED = 12
NUM_LAYERS = 3
MODELSDIR = Path(f"./trained_models/seed_{SEED}/{NUM_LAYERS}layers")

ATTNDIR = Path(f"Results/rollout_local/seed_{SEED}/{NUM_LAYERS}layers")
ATTNDIR.mkdir(parents=True, exist_ok=True)

ROLLDIR = Path(f"Results/rollout_global/seed_{SEED}/{NUM_LAYERS}layers")
ROLLDIR.mkdir(parents=True, exist_ok=True)

### Load model

In [4]:
IN_DIM = X_test.shape[1]

EMB_DIM = 64
NHEAD = 4
FF_DIM = 128
DROPOUT = 0.1

model = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                dropout=DROPOUT).to(DEVICE)

state_dict = torch.load(MODELSDIR / f"TR_model_{NUM_LAYERS}layers.pt")
model.load_state_dict(state_dict)

<All keys matched successfully>

### Extract attention and value matrices

In [5]:
attn_matrices, values_matrices = extract_attention(model, test_loader, device=DEVICE)
attn_matrices_dict = process_attention(attn_matrices)

with open(ATTNDIR / f"attn_matrices{NUM_LAYERS}layers.pkl", "wb") as f:
    pickle.dump(attn_matrices_dict, f)

with open(ATTNDIR / f"value_matrices{NUM_LAYERS}layers.pkl", "wb") as f:
    pickle.dump(values_matrices, f)

### Compute rollout

In [6]:
results = compute_rollout_over_dataloader(model=model, 
                                          dataloader=test_loader, 
                                          device=DEVICE, 
                                          add_residual=True, 
                                          residual_weight=1.0, 
                                          head_fusion="mean", 
                                          normalize=True)

rollout_mean = results['rollout_mean'].numpy()
feature_importance_mean = rollout_mean.sum(axis=0)

rollout = results['rollout'].cpu().numpy()
feature_importance = rollout.sum(axis=1)

np.savez_compressed(ROLLDIR / f"rollout{NUM_LAYERS}layers.npz", 
                    rollout=rollout_mean, 
                    importance=feature_importance_mean)